### Human In The Loop:

In [ ]:
import os
from langchain.chat_models import init_chat_model
llm = init_chat_model("groq:llama3-8b-8192")
llm

In [ ]:
from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.types import Command, interrupt

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

@tool
def human_assistant(query:str) -> str:
    """Request human assistance for the given query."""
    human_response = interrupt({"query": query})
    return human_response["data"]

tool = TavilySearch(max_results=2)
tools = [tool, human_assistant]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state:State):
    message =llm_with_tools.invoke(state["messages"])
    #because we will be interrupting during tool execution,
    #we disable parralell tool calling to avoid repeating any tool invocation when we resume.

    return{"messages":[message]}

graph_builder.add_node("chatbot",chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools",tool_node)

graph_builder.add_conditional_edges("chatbot",tools_condition)

graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START,"chatbot")



In [ ]:
memory = MemorySaver()
graph = graph_builder.compile(checkpointer= memory)

In [1]:
#visualize the graph:
from IPython.display import Image, display

try:
   display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
   #this requires some extra depedency and is optional.
   pass

In [ ]:
user_input = "I need some expert guidance to build AI agents and Could you request assistance for me?"
config = {"configurable": {"thread_id":1}}

events= graph.stream(
    {"message": user_input}, 
    config,
    stream_mode ="values"
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
human_response = (
    "we, the experts are here to help you build AI agents. Let's start by discussing your requirements and the specific areas where you need assistance." 
    "its much more realiable and expensive than simple autonoumous agents, but it will be worth it in the long run. We can provide guidance on architecture, best practices, and even help you with code reviews and testing."
)

human_command = Command(resume={"data": human_response})

events= graph.stream(human_command, config, stream_mode ="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()